# SPIN 7B — Step 2 (AUTHOR-SAMPLE, provisoire): HSIC + perplexity

Throwaway A/B copy of the faithful `7B testing/SPIN_7B_step2_metrics.ipynb`. **Identical metrics code.**
Only two cosmetic differences so it doesn't touch the 7B folder or its outputs:
- reads **`coupled_indices_authorsample.json`** (Step 1 author-sample output),
- writes **`metrics_7b_authorsample.json`** (won't overwrite `metrics_7b.json`).

Add `coupled_indices_authorsample.json` as the Kaggle input, run, then compare the printed HSIC/ppl
against the faithful `metrics_7b.json` (HSIC −0.5%→−7.6% per sigma; ppl 7.602→7.825, +2.9%).


## Step 0 — env + config + find coupled indices

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, gc, time, json   # json: needed under papermill (fresh kernel)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
HSIC_N=300
PPL_SEQLEN=2048
SIGMAS=[50,100,200,400]    # authors' Gaussian-kernel sigma grid (Appendix F: search [50,400], Fig 2 uses 50)
USE_CHAT_TEMPLATE=False     # authors' raw-question protocol (verified)
DTYPE=torch.float16

def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
CP=find('coupled_indices_authorsample.json'); assert CP, 'add coupled_indices_authorsample.json (author-sample Step 1 output) as input'
meta=json.load(open(CP))
COUPLED=meta['coupled']    # {matrix_name: [flat indices]} at the authors' p=q=5e-7
NZ=sum(len(v) for v in COUPLED.values())
print('coupled_indices.json:', CP, '| matrices', len(COUPLED), '| coupled weights', NZ)

## Step 1 — data + model (split)

In [ ]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
fair_q=pd.read_csv(f'{DATA}/salad_fairness.csv')['questions'].tolist()[:HSIC_N]
priv_q=pd.read_csv(f'{DATA}/salad_privacy.csv')['questions'].tolist()[:HSIC_N]
from datasets import load_dataset
_wt=load_dataset('wikitext','wikitext-2-raw-v1',split='test')['text']

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
wt_ids=tokenizer('\n\n'.join(_wt), return_tensors='pt').input_ids
print('model split loaded | entry', GEN_DEV)

## Step 2 — functions (raw reps, frozen-sigma HSIC, perplexity, suppress-from-indices)

In [ ]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def reps(queries,bs=4):
    out=[]
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        h=model(**enc,output_hidden_states=True).hidden_states[-1]; m=enc.attention_mask.unsqueeze(-1)
        out.append(((h*m).sum(1)/m.sum(1)).float().cpu().numpy())
        print(f'    reps {min(i+bs,len(queries))}/{len(queries)}',end='\r')
    print(); return np.concatenate(out,0)
@torch.no_grad()
def reps_all(queries,bs=4):
    # mean-pooled hidden state at EVERY layer (embeddings + each block) -> list of [N,H] arrays
    acc=None
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        hs=model(**enc,output_hidden_states=True).hidden_states   # tuple len = n_layers+1
        m=enc.attention_mask.unsqueeze(-1)
        pooled=[((h*m).sum(1)/m.sum(1)).float().cpu().numpy() for h in hs]
        acc=[[p] for p in pooled] if acc is None else [acc[l]+[pooled[l]] for l in range(len(pooled))]
        print(f'    reps_all {min(i+bs,len(queries))}/{len(queries)}',end='\r')
    print(); return [np.concatenate(a,0) for a in acc]
def _d2(A,B):
    sa=(A**2).sum(1,keepdims=True); sb=(B**2).sum(1,keepdims=True); return np.clip(sa+sb.T-2*A@B.T,0,None)
def hsic(X,Y,sigma):
    # authors' HSIC: tr(K H L H)/(n-1)^2 with Gaussian kernel exp(-||x-y||^2/(2 sigma^2))
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    K=np.exp(-_d2(X,X)/(2*sigma**2)); L=np.exp(-_d2(Y,Y)/(2*sigma**2)); H=np.eye(n)-np.ones((n,n))/n
    return float(np.trace(K@H@L@H)/(n-1)**2)
@torch.no_grad()
def eval_ppl(seqlen=PPL_SEQLEN):
    model.config.use_cache=False; ns=wt_ids.numel()//seqlen; nlls=[]
    for i in range(ns):
        inp=wt_ids[:,i*seqlen:(i+1)*seqlen].to(GEN_DEV); lg=model(inp).logits
        loss=torch.nn.CrossEntropyLoss()(lg[:,:-1,:].reshape(-1,lg.size(-1)),inp[:,1:].reshape(-1))
        nlls.append(loss.float()*seqlen)
        print(f'    ppl {i+1}/{ns}',end='\r')
    print(); return torch.exp(torch.stack(nlls).sum()/(ns*seqlen)).item()
def suppress(coupled):
    d=dict(model.named_parameters()); nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); p.data.view(-1)[idx]=0.0; nz+=len(idxlist)
    return nz
print('functions ready.')

## Step 3 — baseline -> SPIN (frozen sigma so both use the same kernel scale)

In [ ]:
print('baseline: representations (all layers)...')
Lf0=reps_all(fair_q); Lp0=reps_all(priv_q); Xf0,Xp0=Lf0[-1],Lp0[-1]   # [-1] == final layer == original reps()
base_hsic={s:hsic(Xf0,Xp0,s) for s in SIGMAS}
base_hsic_layers={str(s):[hsic(Lf0[l],Lp0[l],s) for l in range(len(Lf0))] for s in SIGMAS}
del Lf0,Lp0; gc.collect()
print('baseline: perplexity...'); base_ppl=eval_ppl()

nz=suppress(COUPLED); print('applied SPIN: zeroed', nz, 'weights')
print('spin: representations (all layers)...'); Lf1=reps_all(fair_q); Lp1=reps_all(priv_q); Xf1,Xp1=Lf1[-1],Lp1[-1]
spin_hsic={s:hsic(Xf1,Xp1,s) for s in SIGMAS}
spin_hsic_layers={str(s):[hsic(Lf1[l],Lp1[l],s) for l in range(len(Lf1))] for s in SIGMAS}
del Lf1,Lp1; gc.collect()
print('spin: perplexity...'); spin_ppl=eval_ppl()

res={'model':MODEL_ID,'p':meta.get('p'),'q':meta.get('q'),'coupled_zeroed':int(nz),
     'sigmas':SIGMAS,'baseline_hsic':base_hsic,'spin_hsic':spin_hsic,
     'n_layers':len(base_hsic_layers[str(SIGMAS[0])]),
     'baseline_hsic_per_layer':base_hsic_layers,'spin_hsic_per_layer':spin_hsic_layers,
     'baseline_ppl':base_ppl,'spin_ppl':spin_ppl}
json.dump(res, open('/kaggle/working/metrics_7b_authorsample.json','w'), indent=2)
print('\n'+'='*62)
print('  HSIC (fairness<->privacy dependence) across the authors sigma grid')
print('='*62)
print(f'{"sigma":>8}{"baseline":>14}{"SPIN":>14}{"change":>12}')
print('-'*62)
for s in SIGMAS:
    b,sp=base_hsic[s],spin_hsic[s]; ch=100*(sp-b)/b if b else 0.0
    print(f'{s:>8}{b:>14.6f}{sp:>14.6f}{ch:>+11.1f}%')
print('-'*62)
print(f'WikiText-2 perplexity: {base_ppl:.3f} -> {spin_ppl:.3f}  ({spin_ppl-base_ppl:+.3f})')
print('HSIC should DROP at each sigma (decoupling, per the paper); perplexity should stay close.')
_sig=200 if 200 in SIGMAS else SIGMAS[0]
print(f'\nper-layer HSIC at sigma={_sig} (base -> SPIN):')
_bl=base_hsic_layers[str(_sig)]; _sl=spin_hsic_layers[str(_sig)]
for l in range(len(_bl)):
    _ch=100*(_sl[l]-_bl[l])/_bl[l] if _bl[l] else 0.0
    print(f'  L{l:2d}: {_bl[l]:.6f} -> {_sl[l]:.6f}  ({_ch:+.1f}%)')
print('saved metrics_7b_authorsample.json  (now incl. per-layer HSIC)')